# Notebook 1 — compare image evidence models

**Question:** Which model labels LADI v2 aerial disaster images usefully enough to derive a *provisional* scheduler priority? LADI visual annotations are **not response urgency ground truth**. Review the contact sheet and output table before setting `SELECTED_MODEL`. A score is not a calibrated urgency probability.

Model roles: CLIP and SigLIP compare text descriptions; LADI small/large are domain classifiers; DeiT tiny is a generic ImageNet control; Grounding DINO finds prompted objects. Their raw scores are not directly comparable. Grounding DINO absence is not evidence of a normal scene.

Priority mapping: 0 Critical, 1 Emergency, 2 Moderate, 3 Normal. It is an editable hypothesis. The human priority column stays blank until you annotate it.

In [ ]:
# Set REPO_URL only in a temporary Colab/Kaggle runtime. Local Jupyter uses the existing checkout.
REPO_URL = None  # e.g. "https://github.com/your-name/SemanticSchedulerEnd2End.git"
REPO_COMMIT = None  # Pin a commit for repeatable cloud runs; None checks out remote HEAD.
PERSISTENT_RESULTS = None  # REQUIRED in cloud: mounted Drive/Kaggle persistent output path.

import os, sys, subprocess, shutil
from pathlib import Path
def cloud_runtime():
    return 'COLAB_GPU' in os.environ or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
if cloud_runtime():
    if not REPO_URL or not PERSISTENT_RESULTS:
        raise ValueError('Set REPO_URL and PERSISTENT_RESULTS before running in Colab/Kaggle')
    checkout = Path('/tmp/semantic_scheduler_pilot_checkout').resolve()
    assert checkout == Path('/tmp/semantic_scheduler_pilot_checkout').resolve()
    if checkout.exists():
        shutil.rmtree(checkout)  # Only this fixed disposable runtime checkout is reset.
    subprocess.run(['git', 'clone', REPO_URL, str(checkout)], check=True)
    if REPO_COMMIT:
        subprocess.run(['git', '-C', str(checkout), 'checkout', REPO_COMMIT], check=True)
    PROJECT_ROOT = checkout
else:
    here = Path.cwd().resolve()
    PROJECT_ROOT = next((p for p in [here, *here.parents] if (p/'PROJECT_MAP.md').is_file()), None)
    if PROJECT_ROOT is None:
        raise RuntimeError('Start Jupyter from this repository or a subdirectory')
sys.path.insert(0, str(PROJECT_ROOT))
RESULTS = Path(PERSISTENT_RESULTS).expanduser().resolve() if PERSISTENT_RESULTS else PROJECT_ROOT/'results'/'scheduler_pilot'
RESULTS.mkdir(parents=True, exist_ok=True)
MODEL_ROOT = PROJECT_ROOT/'models'/'priority_classifiers'
try:
    COMMIT = subprocess.check_output(['git','-C',str(PROJECT_ROOT),'rev-parse','HEAD'], text=True).strip()
except Exception:
    COMMIT = 'unavailable'
print('Repository:', PROJECT_ROOT, 'Commit:', COMMIT, 'Results:', RESULTS)


In [ ]:
# Run once per fresh runtime. Models are downloaded solely by the helper script.
# Optional installs on a clean Colab/Kaggle runtime:
# %pip install -q "transformers>=4.45" "huggingface_hub>=0.25" datasets pillow pandas matplotlib torch sentencepiece protobuf
MODEL_KEYS = ['clip','ladi_small','deit_tiny','siglip','ladi_large','grounding_dino']
DOWNLOAD_MODELS = True  # Downloads missing files into the dedicated model folder; reruns resume.
if DOWNLOAD_MODELS:
    subprocess.run([sys.executable, str(PROJECT_ROOT/'scripts'/'download_priority_models.py'), '--models', *MODEL_KEYS], check=True)
from SemParam.priority_models import MODEL_SPECS, PriorityModel, PRIORITY_NAMES
print({key: spec.repository for key, spec in MODEL_SPECS.items()})


## Fixed pilot sample

Use LADI v2 `train` split for a qualitative demonstration; this is **not an accuracy estimate** because the reference models have seen the training split. The same first 15 photos feed every model. A fixed first-15 sample avoids the excessive memory required to shuffle LADI's large image row groups. Downloaded images are saved to persistent results for Notebook 2. For later quantitative evaluation, use the held-out validation/test split with verified labels. If your local dataset is already available, point `LOCAL_IMAGE_DIR` there; otherwise use the official Hugging Face dataset.

In [ ]:
from PIL import Image
import random, json, platform
import pandas as pd
PILOT_COUNT = 15
SEED = 42
LOCAL_IMAGE_DIR = None  # Optional local folder of LADI v2 images.
image_dir = RESULTS/'pilot_images'
image_dir.mkdir(exist_ok=True)
manifest_path = RESULTS/'pilot_manifest.csv'
if LOCAL_IMAGE_DIR:
    paths = sorted(p for p in Path(LOCAL_IMAGE_DIR).rglob('*') if p.suffix.lower() in ('.jpg','.jpeg','.png'))
    if len(paths) < PILOT_COUNT:
        raise ValueError(f'Need {PILOT_COUNT} images, found {len(paths)}')
    chosen = random.Random(SEED).sample(paths, PILOT_COUNT)
    records = []
    for i, src in enumerate(chosen):
        dest = image_dir/f'pilot_{i:03d}.jpg'
        Image.open(src).convert('RGB').save(dest, quality=92)
        records.append({'image_id': f'pilot_{i:03d}', 'image_path': str(dest), 'source':str(src), 'split':'local', 'human_priority_class':None})
else:
    from datasets import load_dataset
    dataset = load_dataset('MITLL/LADI-v2-dataset', split='train', streaming=True)
    # LADI's large Parquet row groups make shuffle buffering very memory-heavy.
    # A fixed first-15 pilot is reproducible and sufficient for visual comparison.
    sample = dataset.take(PILOT_COUNT)
    records = []
    for i, row in enumerate(sample):
        image = row.get('image')
        if image is None:
            raise KeyError(f'Dataset row fields: {list(row)}; update image field after checking dataset card')
        dest = image_dir/f'pilot_{i:03d}.jpg'
        image.convert('RGB').save(dest, quality=92)
        records.append({'image_id': f'pilot_{i:03d}', 'image_path': str(dest), 'source':str(row.get('id', i)), 'split':'train', 'human_priority_class':None})
manifest = pd.DataFrame(records)
manifest.to_csv(manifest_path, index=False)
print('Pilot images:', len(manifest), 'Manifest:', manifest_path)


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for ax, row in zip(axes.flat, manifest.itertuples()):
    ax.imshow(Image.open(row.image_path)); ax.set_title(row.image_id); ax.axis('off')
fig.tight_layout(); fig.savefig(RESULTS/'pilot_contact_sheet.png', dpi=150); plt.show()


In [ ]:
# Run every model whose local files exist. Missing/failing models remain visible in the result table.
comparison = []
for key in MODEL_KEYS:
    try:
        model = PriorityModel(key, MODEL_ROOT)
    except Exception as exc:
        comparison.append({'model_key': key, 'image_id': None, 'status': 'load_failed', 'error':repr(exc)})
        print(key, 'load failed:', exc)
        continue
    for row in manifest.itertuples():
        try:
            output = model.predict(Image.open(row.image_path).convert('RGB'))
            comparison.append({'image_id':row.image_id,'image_path':row.image_path,'status':'ok',
                               **{k:(json.dumps(v) if k=='native_output' else v) for k,v in output.items()}})
        except Exception as exc:
            comparison.append({'model_key':key,'image_id':row.image_id,'image_path':row.image_path,
                               'status':'inference_failed','error':repr(exc)})
    del model
    print(key, 'finished')
    pd.DataFrame(comparison).to_csv(RESULTS/'pilot_model_comparison.partial.csv', index=False)
comparison_df = pd.DataFrame(comparison)
comparison_df.to_csv(RESULTS/'pilot_model_comparison.csv', index=False)
display(comparison_df.reindex(columns=['image_id','model_key','status','scene_label','priority_name','top_score','error']))
display(comparison_df.reindex(columns=['image_id','model_key','native_output']))


## Manual review and final choice

Inspect the images, `native_output`, and any failures. Record human reference priorities in `pilot_manifest.csv` separately if desired. Set `SELECTED_MODEL` below after review. The final cell runs **only that model** over the chosen dataset and writes a CSV. An unknown mapping remains blank, never silently treated as Normal.

In [ ]:
SELECTED_MODEL = None  # Choose one: clip, siglip, ladi_small, ladi_large, deit_tiny, grounding_dino
FULL_DATASET = False  # False: 15-photo pilot; True: stream the entire LADI train split.
if SELECTED_MODEL is not None:
    if SELECTED_MODEL not in MODEL_SPECS:
        raise ValueError(f'Unknown model {SELECTED_MODEL}')
    chosen_model = PriorityModel(SELECTED_MODEL, MODEL_ROOT)
    destination = RESULTS/f'{SELECTED_MODEL}_priority_labels.csv'
    import csv
    fields = ['image_id','image_path','split','model_key','model_repository','scene_label',
              'priority_class','priority_name','top_score','native_output','score_note','status','error','git_commit']
    if FULL_DATASET:
        from datasets import load_dataset
        source_rows = load_dataset('MITLL/LADI-v2-dataset', split='train', streaming=True)
        iterator = ((str(row.get('id', i)), row['image'].convert('RGB'), '', 'train') for i,row in enumerate(source_rows))
    else:
        iterator = ((row.image_id, Image.open(row.image_path).convert('RGB'), row.image_path, row.split)
                    for row in manifest.itertuples())
    with destination.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields); writer.writeheader()
        count = 0
        for image_id, image, image_path, split in iterator:
            record = dict.fromkeys(fields)
            record.update(image_id=image_id,image_path=image_path,split=split,model_key=SELECTED_MODEL,
                          model_repository=MODEL_SPECS[SELECTED_MODEL].repository,git_commit=COMMIT)
            try:
                output = chosen_model.predict(image)
                record.update(**{k:json.dumps(v) if k=='native_output' else v for k,v in output.items()})
                record['status']='ok'
            except Exception as exc:
                record['status']='inference_failed'; record['error']=repr(exc)
            writer.writerow(record); count += 1
            if count % 100 == 0: stream.flush(); print('Processed', count)
    print('Wrote', count, 'rows to', destination)
else:
    print('Review comparison, then set SELECTED_MODEL and rerun this cell.')
